# 🟡 Day 4 — Connect OpenAI & Chat Messages

**LangGraph from Zero to Advanced · Day 4 of 16**

**Today's goal:** call a real LLM (OpenAI) safely with a `.env` file, understand chat **messages**, and put an LLM **inside a graph node**.

| Lesson | Topic |
|---|---|
| 1 | Put your API key in `.env` |
| 2 | Your first LLM call |
| 3 | Messages: system, human, AI |
| 4 | LLMs have no memory |
| 5 | Stream the answer word by word |
| 6 | An LLM inside a graph node |
| 7 | Mini project: "Explain like I'm 10" pipeline |
| 8 | Practice |

💰 **Cost:** today uses `gpt-4.1-nano` — the whole notebook costs well under ₹1.

## Lesson 1 — Put your API key in `.env`

**Never** paste an API key directly into a notebook (it ends up on GitHub!). Instead:

1. In the **repo root folder**, copy `.env.example` → `.env`
2. Open `.env` and paste your key:

```
OPENAI_API_KEY=sk-...your-key...
OPENAI_MODEL=gpt-4.1-nano
```

3. `.env` is listed in `.gitignore`, so it is never committed. ✅

`python-dotenv` reads that file and puts the values into environment variables. `find_dotenv(usecwd=True)` searches the current folder **and its parents**, so it finds the `.env` in the repo root from any `DayXX` folder.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Copy .env.example to .env in the repo root and add your key.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
print("✅ API key loaded. Model:", MODEL)

## Lesson 2 — Your first LLM call

`ChatOpenAI` is LangChain's wrapper around OpenAI chat models. `temperature=0` makes answers more stable (less random).

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

response = llm.invoke("In one sentence, what is LangGraph?")
print(type(response).__name__)   # AIMessage
print(response.content)

The result is an **`AIMessage`**. Besides `.content` it carries useful info, such as token usage:

In [ ]:
print("Token usage:", response.usage_metadata)

## Lesson 3 — Messages: system, human, AI

Chat models take a **list of messages**. Each message has a role:

| Class | `.type` | Who writes it | Used for |
|---|---|---|---|
| `SystemMessage` | `system` | you (developer) | rules, personality |
| `HumanMessage` | `human` | the user | questions |
| `AIMessage` | `ai` | the model | answers |
| `ToolMessage` | `tool` | your code | tool results (Day 6) |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a cricket commentator. Answer in 2 short lines, with excitement."),
    HumanMessage(content="Explain what a Python function is."),
]
reply = llm.invoke(messages)
reply.pretty_print()

💡 Shortcut: you can write messages as tuples — `("system", "...")`, `("user", "...")`, `("assistant", "...")`.

In [ ]:
reply = llm.invoke([
    ("system", "Reply only with an emoji."),
    ("user", "How do you feel about Mondays?"),
])
print(reply.content)

## Lesson 4 — LLMs have no memory

Each call is **independent**. The model remembers nothing unless *you* send the earlier messages again.

In [ ]:
llm.invoke([HumanMessage(content="Hi! My name is Asha.")])
r = llm.invoke([HumanMessage(content="What is my name?")])
print("Without history:", r.content)

In [ ]:
history = [
    HumanMessage(content="Hi! My name is Asha."),
    AIMessage(content="Nice to meet you, Asha!"),
    HumanMessage(content="What is my name?"),
]
r = llm.invoke(history)
print("With history:", r.content)

🔑 **Key insight:** a chatbot = a growing **list of messages** sent to the model every time. Tomorrow LangGraph will manage that list for us.

## Lesson 5 — Stream the answer word by word

`llm.stream(...)` returns small **chunks** as they are generated — like ChatGPT typing.

In [ ]:
for chunk in llm.stream("Count from 1 to 10 in words, separated by spaces."):
    print(chunk.content, end="", flush=True)
print()

## Lesson 6 — An LLM inside a graph node

An LLM call is just Python — so it fits inside a node. Same 5-step recipe as Day 1!

```
START ──► answer ──► END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

class QAState(TypedDict):
    question: str
    answer: str

def answer(state: QAState) -> dict:
    msg = llm.invoke([
        SystemMessage(content="You are a helpful teacher. Answer in at most 3 sentences."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": msg.content}

g = StateGraph(QAState)
g.add_node("answer", answer)
g.add_edge(START, "answer")
g.add_edge("answer", END)
qa_app = g.compile()

result = qa_app.invoke({"question": "Why is the sky blue?"})
print(result["answer"])

In [ ]:
assert isinstance(result["answer"], str) and len(result["answer"]) > 0
print("✅ Your first LLM graph works!")

## Lesson 7 — Mini project: "Explain like I'm 10"

Two LLM nodes in a row — the second improves the first:

```
START ──► explain ──► simplify ──► END
```

In [ ]:
class ExplainState(TypedDict):
    topic: str
    explanation: str
    simple: str

def explain(state):
    msg = llm.invoke(f"Explain '{state['topic']}' in 4 sentences for an engineer.")
    return {"explanation": msg.content}

def simplify(state):
    msg = llm.invoke(
        "Rewrite this so a 10-year-old understands it. Use one fun example. "
        f"Max 3 sentences.\n\n{state['explanation']}"
    )
    return {"simple": msg.content}

g = StateGraph(ExplainState)
g.add_sequence([explain, simplify])
g.add_edge(START, "explain")
g.add_edge("simplify", END)
eli10 = g.compile()
show_graph(eli10)

In [ ]:
out = eli10.invoke({"topic": "neural networks"})
print("🧑‍💻 ENGINEER:\n", out["explanation"])
print("\n🧒 KID:\n", out["simple"])

Use `stream_mode="updates"` to watch each LLM node finish:

In [ ]:
for step in eli10.stream({"topic": "blockchain"}, stream_mode="updates"):
    for node, update in step.items():
        print(f"✅ {node} finished → keys: {list(update)}")

## Lesson 8 — Practice

**Exercise 1.** Make a `translate` graph: state has `text` and `language`; one node asks the LLM to translate `text` into `language`, and stores it in `translation`. Try Hindi and Marathi.

**Exercise 2.** Build `START → write_tweet → add_hashtags → END`. The first node writes a tweet about a `topic`; the second asks the LLM to add 3 hashtags.

In [ ]:
# ✅ Solution 1
class TranslateState(TypedDict):
    text: str
    language: str
    translation: str

def translate(state):
    msg = llm.invoke([
        SystemMessage(content="You are a translator. Reply with only the translation."),
        HumanMessage(content=f"Translate into {state['language']}: {state['text']}"),
    ])
    return {"translation": msg.content}

g = StateGraph(TranslateState)
g.add_node("translate", translate)
g.add_edge(START, "translate")
g.add_edge("translate", END)
translator = g.compile()

for lang in ("Hindi", "Marathi"):
    print(lang, "→", translator.invoke({"text": "Learning never stops.", "language": lang})["translation"])

In [ ]:
# ✅ Solution 2
class TweetState(TypedDict):
    topic: str
    tweet: str

def write_tweet(state):
    return {"tweet": llm.invoke(f"Write one short tweet (max 200 chars) about {state['topic']}. No hashtags.").content}

def add_hashtags(state):
    return {"tweet": llm.invoke(f"Add exactly 3 relevant hashtags at the end of this tweet. Return the full tweet only:\n{state['tweet']}").content}

g = StateGraph(TweetState)
g.add_sequence([write_tweet, add_hashtags])
g.add_edge(START, "write_tweet")
g.add_edge("add_hashtags", END)
print(g.compile().invoke({"topic": "learning LangGraph"})["tweet"])

## 🎯 Day 4 Recap

* API key lives in `.env` → `load_dotenv(find_dotenv(usecwd=True))`.
* `ChatOpenAI(model=..., temperature=0)` → `.invoke()` returns an `AIMessage`.
* Messages: `SystemMessage` (rules), `HumanMessage` (user), `AIMessage` (model).
* LLMs are **stateless** — you must send the history.
* An LLM call is just code inside a node.

**Tomorrow (Day 5):** build a real **chatbot** graph with `MessagesState` and the `add_messages` reducer.